In [ ]:
# Activar matplotlib inline para Jupyter
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
from matplotlib import rcParams

# ==============================================================================
# 1. CONFIGURACIÓN GENERAL (¡Aquí cambias lo que necesites!)
# ==============================================================================
nombre_archivo = 'data_cartesianas'  # Nombre de tu archivo de datos

# CONTROL DE VELOCIDAD DE REPRODUCCIÓN:
# 1.0 = Tiempo real (5s de datos duran 5s de video)
# 2.0 = Cámara rápida (el doble de rápido, ideal si tus datos son muy largos)
# 0.5 = Cámara lenta (la mitad de rápido, ideal para ver el caos en detalle)
factor_velocidad = 1.0 

# CONTROL DE DURACIÓN DE LOS DATOS (Opcional):
# Si tu archivo tiene 10 segundos pero SOLO quieres animar los primeros 3 segundos,
# cambia este valor. Si quieres animar TODO el archivo completo, déjalo en None.
recortar_tiempo_maximo = None  # Ejemplo: 3.0 para detenerse a los 3 segundos

# Ajuste de límite de memoria para animaciones en Jupyter (en MB)
rcParams['animation.embed_limit'] = 100.0 


# ==============================================================================
# 2. LECTURA Y PROCESAMIENTO DE DATOS
# ==============================================================================
print("Leyendo datos...")

def leer_datos(archivo):
    # np.loadtxt detecta espacios y tabuladores automáticamente
    datos = np.loadtxt(archivo) 
    t = datos[:, 0]
    x1 = datos[:, 1]
    y1 = datos[:, 2]
    x2 = datos[:, 3]
    y2 = datos[:, 4]
    return t, x1, y1, x2, y2

t, x1, y1, x2, y2 = leer_datos(nombre_archivo)

# Aplicar recorte de tiempo si el usuario lo especificó arriba
if recortar_tiempo_maximo is not None:
    indices_validos = t <= recortar_tiempo_maximo
    t = t[indices_validos]
    x1 = x1[indices_validos]
    y1 = y1[indices_validos]
    x2 = x2[indices_validos]
    y2 = y2[indices_validos]


# ==============================================================================
# 3. CÁLCULO DE TIEMPOS Y SINCRONIZACIÓN AUTOMÁTICA
# ==============================================================================
duracion_real = t[-1] - t[0] 
total_puntos = len(t)        

# Apuntamos a unos 50 cuadros por segundo (FPS) para que el navegador vaya fluido
fps_objetivo = 50 
frames_necesarios = int(duracion_real * fps_objetivo * factor_velocidad)

# Calculamos el salto de filas necesario para cumplir la meta de FPS
salto = max(1, total_puntos // frames_necesarios)

# Ajustamos las propiedades finales de la animación
frames_animados = range(0, total_puntos, salto)
num_frames_finales = len(frames_animados)

intervalo_jupyter = int((duracion_real * 1000) / (num_frames_finales * factor_velocidad))
fps_video_final = int((num_frames_finales / duracion_real) * factor_velocidad)

print(f"-> Duración de los datos a animar: {duracion_real:.2f} segundos")
print(f"-> Usando 1 de cada {salto} puntos (Total frames: {num_frames_finales})")
print(f"-> FPS del Video final: {fps_video_final} | Intervalo Jupyter: {intervalo_jupyter} ms\n")


# ==============================================================================
# 4. CONFIGURACIÓN DEL GRÁFICO (LIENZO)
# ==============================================================================
fig, ax = plt.subplots(figsize=(6, 6))

# Límites dinámicos basados en los extremos de ambas masas
min_x = min(np.min(x1), np.min(x2)) - 0.5
max_x = max(np.max(x1), np.max(x2)) + 0.5
min_y = min(np.min(y1), np.min(y2)) - 0.5
max_y = max(np.max(y1), np.max(y2)) + 0.5

ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Simulación de Péndulo Doble Elástico')
ax.grid(True, linestyle='--', alpha=0.5)

# Definir el péndulo: 'o-' dibuja líneas azules con círculos rojos en las uniones
linea, = ax.plot([], [], 'o-', linewidth=2, color='b', markersize=8, markerfacecolor='r')

# Texto en pantalla para el tiempo
tiempo_texto = ax.text(0.05, 0.95, '', transform=ax.transAxes)


# ==============================================================================
# 5. FUNCIONES DE ANIMACIÓN Y RENDERIZADO
# ==============================================================================
def animar(i):
    # Coordenadas: Origen (0,0) -> Masa 1 (x1, y1) -> Masa 2 (x2, y2)
    x_actual = [0, x1[i], x2[i]]
    y_actual = [0, y1[i], y2[i]]
    
    linea.set_data(x_actual, y_actual)
    tiempo_texto.set_text(f"t = {t[i]:.2f} s")
    return linea, tiempo_texto

print("Renderizando animación para Jupyter (esto puede tardar unos segundos)...")
ani = animation.FuncAnimation(
    fig, animar, frames=frames_animados, interval=intervalo_jupyter, blit=True
)

# Cerrar la figura estática de matplotlib para que no se duplique abajo
plt.close(fig)

# Mostrar el reproductor interactivo en el cuaderno
display(HTML(ani.to_jshtml()))

print("Guardando archivo de video MP4...")
# Guardar el archivo de video perfectamente sincronizado
ani.save('pendulo_doble_sincronizado.mp4', writer='ffmpeg', fps=fps_video_final, dpi=150)

print("¡Todo listo! El video se guardó como 'pendulo_doble_sincronizado.mp4'")

